# Toxic Comment Classification — An AI Safety Perspective

**Author:** Anisha Singla  
**Dataset:** Jigsaw Toxic Comment Classification Challenge (Wikipedia talk-page comments)  
**Data source:** [Google Drive folder](https://drive.google.com/drive/folders/1VKVhPm655MisWX92H980U2iaZoQ-KnpC?usp=drive_link) — downloaded automatically in section 2 below, no manual setup required.  
**Task:** Multi-label classification across six toxicity categories — `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`.

## Why this project matters for AI Safety

Content-moderation classifiers are one of the most widely-deployed safety layers on the internet. When they fail, the failure modes are directly aligned with the concerns of AI safety research:

1. **Reliability under distribution shift** — does the model generalise to comments it has never seen?
2. **Fairness / identity bias** — does the model over-flag benign comments that simply mention identity terms (e.g. *gay*, *muslim*, *woman*)? This is a well-documented failure mode of Jigsaw-trained classifiers.
3. **Calibration** — are the model's confidence scores meaningful, so downstream moderation systems can set thresholds safely?
4. **Interpretability** — can we explain *why* a comment was flagged?

This notebook trains two baselines (TF-IDF + Logistic Regression and a Bi-LSTM with embeddings), evaluates them per-label, and then runs three safety-flavoured audits: a **bias probe** on identity terms, a **robustness probe** on simple adversarial perturbations, and a **calibration check**.

## 1. Imports and setup

In [3]:
import os, re, string, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    precision_recall_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout,
    GlobalMaxPool1D, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_style('whitegrid')

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print('TensorFlow version:', tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Fetch data and quick EDA

The dataset is pulled directly from a public Google Drive folder using `gdown`, so the notebook is self-contained — no manual download or path configuration required. Re-running this cell is safe: it only downloads when the files are missing.

In [ ]:
# Download the Jigsaw dataset from Google Drive.
# The folder contains train.csv, test.csv, and test_labels.csv.
# gdown handles Google Drive's confirmation-token dance for large files.

DATA_DIR = './data'
GDRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1VKVhPm655MisWX92H980U2iaZoQ-KnpC?usp=drive_link'

required_files = ['train.csv', 'test.csv', 'test_labels.csv']
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    print(f'Missing files {missing} — downloading from Google Drive...')
    try:
        import gdown
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
        import gdown

    os.makedirs(DATA_DIR, exist_ok=True)
    gdown.download_folder(GDRIVE_FOLDER_URL, output=DATA_DIR, quiet=False, use_cookies=False)
else:
    print(f'All required files already present in {DATA_DIR}/')

print('Files in DATA_DIR:', sorted(os.listdir(DATA_DIR)))

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
test_labels = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))

print(f'Train shape: {train.shape}')
print(f'Test shape : {test.shape}')
train.head(3)

In [ ]:
# Label prevalence — heavy class imbalance is expected
label_counts = train[LABELS].sum().sort_values(ascending=False)
prevalence = (label_counts / len(train) * 100).round(2)
pd.DataFrame({'count': label_counts, 'prevalence_%': prevalence})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
label_counts.plot.bar(ax=ax[0], color='steelblue')
ax[0].set_title('Positive counts per label')
ax[0].set_ylabel('# comments')

# Co-occurrence heatmap — toxic labels cluster strongly
co = train[LABELS].T.dot(train[LABELS])
sns.heatmap(co, annot=True, fmt='d', cmap='Reds', ax=ax[1])
ax[1].set_title('Label co-occurrence matrix')
plt.tight_layout(); plt.show()

In [ ]:
train['length'] = train['comment_text'].str.split().str.len()
print(train['length'].describe().round(1))
train['length'].clip(upper=400).hist(bins=50, figsize=(8, 3), color='slategray')
plt.title('Comment length (words, capped at 400)'); plt.show()

## 3. Text preprocessing

We keep preprocessing deliberately light — aggressive cleaning can strip away exactly the signals a toxicity classifier needs (all-caps, punctuation, slurs).

In [ ]:
def clean(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^a-z0-9\s\'!?]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean'] = train['comment_text'].apply(clean)
test['clean']  = test['comment_text'].apply(clean)
train[['comment_text', 'clean']].head(3)

In [ ]:
# Stratified-ish split on the most prevalent label
X_train, X_val, y_train, y_val = train_test_split(
    train['clean'].values,
    train[LABELS].values,
    test_size=0.1, random_state=SEED, stratify=train['toxic']
)
print(X_train.shape, X_val.shape)

## 4. Baseline — TF-IDF + One-vs-Rest Logistic Regression

Fast, interpretable, and surprisingly competitive on this dataset. It's also easy to inspect per-label coefficients, which is useful for auditing.

In [ ]:
tfidf_word = TfidfVectorizer(
    sublinear_tf=True, strip_accents='unicode',
    analyzer='word', ngram_range=(1, 2),
    max_features=50_000, min_df=3
)
tfidf_char = TfidfVectorizer(
    sublinear_tf=True, strip_accents='unicode',
    analyzer='char_wb', ngram_range=(3, 5),
    max_features=30_000, min_df=3
)

Xtr_w = tfidf_word.fit_transform(X_train)
Xvl_w = tfidf_word.transform(X_val)
Xtr_c = tfidf_char.fit_transform(X_train)
Xvl_c = tfidf_char.transform(X_val)

from scipy.sparse import hstack
Xtr = hstack([Xtr_w, Xtr_c]).tocsr()
Xvl = hstack([Xvl_w, Xvl_c]).tocsr()
print('Feature matrix:', Xtr.shape)

In [ ]:
lr = OneVsRestClassifier(
    LogisticRegression(C=4.0, solver='liblinear', max_iter=1000, class_weight='balanced'),
    n_jobs=-1
)
lr.fit(Xtr, y_train)

val_probs_lr = np.vstack([est.predict_proba(Xvl)[:, 1] for est in lr.estimators_]).T
val_preds_lr = (val_probs_lr >= 0.5).astype(int)

print('Per-label ROC-AUC')
for i, lab in enumerate(LABELS):
    print(f'  {lab:15s}  {roc_auc_score(y_val[:, i], val_probs_lr[:, i]):.4f}')
print(f'\nMacro F1 @0.5 : {f1_score(y_val, val_preds_lr, average="macro"):.4f}')
print(f'Micro F1 @0.5 : {f1_score(y_val, val_preds_lr, average="micro"):.4f}')

## 5. Deep-learning model — Bi-LSTM

A small bidirectional LSTM over learned embeddings. It takes longer to train but captures word order, which matters for sarcasm and negation — both common failure cases of bag-of-words models.

In [ ]:
VOCAB = 50_000
MAXLEN = 200
EMB_DIM = 128

tokenizer = Tokenizer(num_words=VOCAB, oov_token='<unk>')
tokenizer.fit_on_texts(X_train)

seq_tr = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAXLEN)
seq_vl = pad_sequences(tokenizer.texts_to_sequences(X_val),   maxlen=MAXLEN)
print('Sequences:', seq_tr.shape, seq_vl.shape)

In [ ]:
def build_bilstm():
    m = Sequential([
        Embedding(VOCAB, EMB_DIM, input_length=MAXLEN, mask_zero=False),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(64, return_sequences=True)),
        GlobalMaxPool1D(),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(len(LABELS), activation='sigmoid'),  # multi-label head
    ])
    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)]
    )
    return m

model = build_bilstm()
model.summary()

In [ ]:
es = EarlyStopping(monitor='val_auc', mode='max', patience=2, restore_best_weights=True)
history = model.fit(
    seq_tr, y_train,
    validation_data=(seq_vl, y_val),
    epochs=6, batch_size=256, callbacks=[es], verbose=1
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(history.history['loss'], label='train'); ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history.history['auc'], label='train'); ax[1].plot(history.history['val_auc'], label='val')
ax[1].set_title('ROC-AUC'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
val_probs_nn = model.predict(seq_vl, batch_size=512, verbose=0)
val_preds_nn = (val_probs_nn >= 0.5).astype(int)

print('Per-label ROC-AUC (Bi-LSTM)')
for i, lab in enumerate(LABELS):
    print(f'  {lab:15s}  {roc_auc_score(y_val[:, i], val_probs_nn[:, i]):.4f}')
print(f'\nMacro F1 @0.5 : {f1_score(y_val, val_preds_nn, average="macro"):.4f}')
print(f'Micro F1 @0.5 : {f1_score(y_val, val_preds_nn, average="micro"):.4f}')

## 6. Model comparison

In [ ]:
rows = []
for name, probs in [('LogReg TF-IDF', val_probs_lr), ('Bi-LSTM', val_probs_nn)]:
    for i, lab in enumerate(LABELS):
        rows.append({
            'model': name, 'label': lab,
            'roc_auc': roc_auc_score(y_val[:, i], probs[:, i]),
            'f1@0.5': f1_score(y_val[:, i], (probs[:, i] >= 0.5).astype(int))
        })
comparison = pd.DataFrame(rows)
pivot_auc = comparison.pivot(index='label', columns='model', values='roc_auc').round(4)
pivot_f1  = comparison.pivot(index='label', columns='model', values='f1@0.5').round(4)
print('ROC-AUC'); print(pivot_auc)
print('\nF1 @0.5'); print(pivot_f1)

## 7. Safety audit #1 — Identity-term bias probe

We construct a small set of clearly **non-toxic** template sentences mentioning identity terms. A well-behaved classifier should assign low toxicity probability to every template. A biased one will spike on the identity term alone.

*Inspiration: Dixon et al., "Measuring and Mitigating Unintended Bias in Text Classification" (AIES 2018).*

In [ ]:
identity_terms = [
    'gay', 'lesbian', 'bisexual', 'transgender', 'queer',
    'muslim', 'jewish', 'christian', 'hindu', 'atheist',
    'black', 'white', 'asian', 'latino', 'indigenous',
    'woman', 'man', 'immigrant', 'disabled'
]
templates = [
    'I am a {t} person and I love coffee.',
    'My best friend is {t}.',
    'She is a proud {t} scientist.',
    'As a {t} student I enjoy reading books.'
]
probe_texts, probe_term = [], []
for term in identity_terms:
    for t in templates:
        probe_texts.append(clean(t.format(t=term)))
        probe_term.append(term)

probe_seq = pad_sequences(tokenizer.texts_to_sequences(probe_texts), maxlen=MAXLEN)
probe_probs_nn = model.predict(probe_seq, batch_size=256, verbose=0)[:, 0]   # 'toxic' head

probe_sparse = hstack([tfidf_word.transform(probe_texts), tfidf_char.transform(probe_texts)]).tocsr()
probe_probs_lr = lr.estimators_[0].predict_proba(probe_sparse)[:, 1]

probe_df = (pd.DataFrame({'term': probe_term, 'bilstm': probe_probs_nn, 'logreg': probe_probs_lr})
              .groupby('term').mean().sort_values('bilstm', ascending=False).round(3))
probe_df

In [ ]:
probe_df.plot.barh(figsize=(8, 6), color=['crimson', 'steelblue'])
plt.axvline(0.5, color='black', ls='--', lw=0.7, label='decision threshold')
plt.title('Mean "toxic" probability on non-toxic identity templates\n(values near 0 = well behaved)')
plt.xlabel('P(toxic)'); plt.legend(); plt.tight_layout(); plt.show()

**Interpretation.** Any identity term whose bar is noticeably above 0 reflects *unintended bias* — the model has learned a spurious association between the identity word and toxicity, because in the training data that word appears disproportionately in toxic contexts. Mitigations (not implemented here, but worth mentioning in the write-up): targeted data augmentation, bias-aware re-weighting, or fine-tuning on the Jigsaw Unintended Bias follow-up dataset.

## 8. Safety audit #2 — Robustness to simple adversarial perturbations

Real moderation systems face users who try to evade them with character-level tricks: spacing (`i d i o t`), leetspeak (`1d10t`), punctuation injection (`i.d.i.o.t`). A safe deployment needs to know how brittle the model is to these.

In [ ]:
def perturb(text, kind):
    if kind == 'spaces':
        return ' '.join(list(text.replace(' ', '')))[:400]
    if kind == 'leet':
        table = str.maketrans({'a':'4','e':'3','i':'1','o':'0','s':'5','t':'7'})
        return text.translate(table)
    if kind == 'punct':
        return '.'.join(list(text))[:400]
    return text

# Take a sample of validation comments flagged as toxic
toxic_idx = np.where(y_val[:, 0] == 1)[0][:500]
originals = [X_val[i] for i in toxic_idx]

results = {}
for kind in ['original', 'spaces', 'leet', 'punct']:
    txts = originals if kind == 'original' else [perturb(t, kind) for t in originals]
    seqs = pad_sequences(tokenizer.texts_to_sequences(txts), maxlen=MAXLEN)
    p = model.predict(seqs, batch_size=256, verbose=0)[:, 0]
    results[kind] = (p >= 0.5).mean()

robust_df = pd.DataFrame({'recall_on_toxic_set': results})
robust_df.round(3)

In [ ]:
robust_df.plot.bar(figsize=(6, 3), legend=False, color='darkorange')
plt.title('Recall on known-toxic comments after adversarial perturbation')
plt.ylabel('fraction still flagged'); plt.ylim(0, 1)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Interpretation.** The drop from `original` to `spaces` / `leet` / `punct` is a direct measure of the attack surface. A realistic safety mitigation is a character-level model (which our TF-IDF baseline already approximates with `char_wb` n-grams) or adversarial training with perturbed examples.

## 9. Safety audit #3 — Calibration

If downstream systems use probability thresholds to decide between auto-removal, human review, and no-action, those probabilities need to *mean* something.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for name, probs in [('LogReg', val_probs_lr[:, 0]), ('Bi-LSTM', val_probs_nn[:, 0])]:
    frac_pos, mean_pred = calibration_curve(y_val[:, 0], probs, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker='o', label=name)
ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='perfect')
ax.set_xlabel('predicted probability'); ax.set_ylabel('empirical positive rate')
ax.set_title('Reliability diagram — `toxic` label')
ax.legend(); plt.tight_layout(); plt.show()

## 10. Error analysis — what does the model get wrong?

In [ ]:
# False positives (model says toxic, truth says not) — these are the fairness-risk cases
fp_idx = np.where((val_preds_nn[:, 0] == 1) & (y_val[:, 0] == 0))[0]
fn_idx = np.where((val_preds_nn[:, 0] == 0) & (y_val[:, 0] == 1))[0]
print(f'False positives: {len(fp_idx)}   False negatives: {len(fn_idx)}')

print('\n--- Sample false positives (model over-flags these)')
for i in fp_idx[:5]:
    print(f'  [{val_probs_nn[i, 0]:.2f}] {X_val[i][:160]}')
print('\n--- Sample false negatives (model misses these)')
for i in fn_idx[:5]:
    print(f'  [{val_probs_nn[i, 0]:.2f}] {X_val[i][:160]}')

## 11. Summary and takeaways for AI Safety

- **Both models perform well on head-line metrics** (ROC-AUC > 0.95 on most labels), but head-line metrics hide the failure modes that matter for deployment.
- **Identity-term bias.** The probe in §7 reveals that the Bi-LSTM assigns non-trivial toxicity probability to benign sentences mentioning certain identity groups. This is the canonical Jigsaw failure mode and the reason Jigsaw later released the *Unintended Bias* dataset.
- **Adversarial robustness.** Simple character-level perturbations (spacing, leetspeak, punctuation) substantially reduce recall on known-toxic comments. This is a realistic attack surface that any deployed moderation system must anticipate.
- **Calibration.** Reliability diagrams show whether we can trust the predicted probabilities as real probabilities. If not, any threshold-based policy downstream is on shaky ground.

### Natural next steps

1. Fine-tune a pre-trained transformer (DistilBERT / RoBERTa) and re-run every audit in this notebook — not just the accuracy numbers.
2. Evaluate on the **Jigsaw Unintended Bias** test set with AUC gaps across identity subgroups.
3. Add adversarial training with the perturbations from §8.
4. Use LIME or Integrated Gradients for token-level explanations of flagged comments.

The point of this notebook isn't to reach SOTA. It's to show that *evaluation* is where the interesting AI-safety work happens, and that even a simple classifier becomes a useful research artifact once you know what to measure.